In [ ]:
# %% [markdown]
# # Sistema de Gestión de Tutorías
# ## Parcial 2 - Programación Orientada a Objetos
#
# Este notebook implementa un sistema completo de gestión de tutorías con:
# - Jerarquía de clases (Herencia)
# - Base de datos SQLite
# - Polimorfismo
# - Interfaz de consola interactiva

# %% [markdown]
# ## 1. Importación de Librerías

# %%
import sqlite3
from datetime import datetime
from typing import List, Optional

# %% [markdown]
# ## 2. Definición de Clases (Herencia y Polimorfismo)

# %%
class Usuario:
    """
    Clase padre que representa un usuario del sistema.
    Atributos comunes a todos los usuarios.
    """
    def __init__(self, id: int, nombre: str, correo: str, rol: str):
        self.id = id
        self.nombre = nombre
        self.correo = correo
        self.rol = rol

    def mostrar_info(self):
        """Método para mostrar información del usuario"""
        return f"ID: {self.id} | Nombre: {self.nombre} | Correo: {self.correo} | Rol: {self.rol}"

    def puede_reservar(self) -> bool:
        """
        Método polimórfico que será sobrescrito por las clases hijas.
        Define si el usuario puede reservar sesiones.
        """
        return False


class Tutor(Usuario):
    """
    Clase hija que representa a un tutor.
    Hereda de Usuario y añade especialidades.
    """
    def __init__(self, id: int, nombre: str, correo: str, especialidades: List[str] = None):
        super().__init__(id, nombre, correo, "tutor")
        self.especialidades = especialidades if especialidades else []

    def agregar_especialidad(self, especialidad: str):
        """Agrega una especialidad al tutor"""
        if especialidad not in self.especialidades:
            self.especialidades.append(especialidad)

    def puede_reservar(self) -> bool:
        """Los tutores NO pueden reservar sesiones (polimorfismo)"""
        return False

    def mostrar_info(self):
        """Sobrescribe el método de la clase padre"""
        info_base = super().mostrar_info()
        especialidades_str = ", ".join(self.especialidades) if self.especialidades else "Ninguna"
        return f"{info_base} | Especialidades: {especialidades_str}"


class Estudiante(Usuario):
    """
    Clase hija que representa a un estudiante.
    Hereda de Usuario.
    """
    def __init__(self, id: int, nombre: str, correo: str):
        super().__init__(id, nombre, correo, "estudiante")

    def puede_reservar(self) -> bool:
        """Los estudiantes SÍ pueden reservar sesiones (polimorfismo)"""
        return True

# %% [markdown]
# ## 3. Clase Gestora de Base de Datos

# %%
class GestorBaseDatos:
    """
    Clase para manejar todas las operaciones de base de datos.
    """
    def __init__(self, nombre_db: str = "tutorias.db"):
        self.nombre_db = nombre_db
        self.crear_tablas()
        self.poblar_materias_iniciales()

    def obtener_conexion(self):
        """Crea y retorna una conexión a la base de datos"""
        return sqlite3.connect(self.nombre_db)

    def crear_tablas(self):
        """Crea las tablas necesarias si no existen"""
        conn = self.obtener_conexion()
        cursor = conn.cursor()

        # Tabla de usuarios
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS usuarios (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                correo TEXT NOT NULL UNIQUE,
                rol TEXT NOT NULL CHECK(rol IN ('tutor', 'estudiante'))
            )
        ''')

        # Tabla de materias
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS materias (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL UNIQUE
            )
        ''')

        # Tabla de sesiones
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS sesiones (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                tutor_id INTEGER NOT NULL,
                materia_id INTEGER NOT NULL,
                fecha_hora TEXT NOT NULL,
                estudiante_id INTEGER,
                estado TEXT NOT NULL CHECK(estado IN ('disponible', 'reservada')),
                FOREIGN KEY (tutor_id) REFERENCES usuarios(id),
                FOREIGN KEY (materia_id) REFERENCES materias(id),
                FOREIGN KEY (estudiante_id) REFERENCES usuarios(id)
            )
        ''')

        conn.commit()
        conn.close()
        print("✓ Base de datos inicializada correctamente")

    def poblar_materias_iniciales(self):
        """Puebla la tabla de materias con datos iniciales si está vacía"""
        conn = self.obtener_conexion()
        cursor = conn.cursor()

        cursor.execute("SELECT COUNT(*) FROM materias")
        if cursor.fetchone()[0] == 0:
            materias_iniciales = [
                "Matemáticas",
                "Física",
                "Química",
                "Programación",
                "Inglés",
                "Estadística",
                "Cálculo",
                "Álgebra Lineal"
            ]

            for materia in materias_iniciales:
                cursor.execute("INSERT INTO materias (nombre) VALUES (?)", (materia,))

            conn.commit()
            print("✓ Materias iniciales cargadas")

        conn.close()

    def registrar_usuario(self, nombre: str, correo: str, rol: str) -> Optional[int]:
        """Registra un nuevo usuario en la base de datos"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            cursor.execute(
                "INSERT INTO usuarios (nombre, correo, rol) VALUES (?, ?, ?)",
                (nombre, correo, rol)
            )

            usuario_id = cursor.lastrowid
            conn.commit()
            conn.close()

            return usuario_id
        except sqlite3.IntegrityError:
            print("✗ Error: El correo ya está registrado")
            return None
        except Exception as e:
            print(f"✗ Error al registrar usuario: {e}")
            return None

    def obtener_usuario(self, usuario_id: int) -> Optional[Usuario]:
        """Obtiene un usuario por su ID y retorna el objeto correspondiente"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            cursor.execute(
                "SELECT id, nombre, correo, rol FROM usuarios WHERE id = ?",
                (usuario_id,)
            )

            resultado = cursor.fetchone()
            conn.close()

            if resultado:
                id_user, nombre, correo, rol = resultado
                if rol == "tutor":
                    return Tutor(id_user, nombre, correo)
                else:
                    return Estudiante(id_user, nombre, correo)

            return None
        except Exception as e:
            print(f"✗ Error al obtener usuario: {e}")
            return None

    def listar_materias(self) -> List[tuple]:
        """Lista todas las materias disponibles"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            cursor.execute("SELECT id, nombre FROM materias ORDER BY nombre")
            materias = cursor.fetchall()

            conn.close()
            return materias
        except Exception as e:
            print(f"✗ Error al listar materias: {e}")
            return []

    def crear_sesion(self, tutor_id: int, materia_id: int, fecha_hora: str) -> bool:
        """Crea una nueva sesión de tutoría"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            cursor.execute(
                """INSERT INTO sesiones (tutor_id, materia_id, fecha_hora, estudiante_id, estado)
                   VALUES (?, ?, ?, NULL, 'disponible')""",
                (tutor_id, materia_id, fecha_hora)
            )

            conn.commit()
            conn.close()
            return True
        except Exception as e:
            print(f"✗ Error al crear sesión: {e}")
            return False

    def listar_sesiones_disponibles(self) -> List[tuple]:
        """Lista todas las sesiones disponibles con información detallada"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            cursor.execute("""
                SELECT s.id, u.nombre, m.nombre, s.fecha_hora
                FROM sesiones s
                JOIN usuarios u ON s.tutor_id = u.id
                JOIN materias m ON s.materia_id = m.id
                WHERE s.estado = 'disponible'
                ORDER BY s.fecha_hora
            """)

            sesiones = cursor.fetchall()
            conn.close()
            return sesiones
        except Exception as e:
            print(f"✗ Error al listar sesiones: {e}")
            return []

    def reservar_sesion(self, sesion_id: int, estudiante_id: int) -> bool:
        """Reserva una sesión para un estudiante"""
        try:
            conn = self.obtener_conexion()
            cursor = conn.cursor()

            # Verificar que la sesión existe y está disponible
            cursor.execute(
                "SELECT estado FROM sesiones WHERE id = ?",
                (sesion_id,)
            )
            resultado = cursor.fetchone()

            if not resultado:
                print("✗ Error: La sesión no existe")
                return False

            if resultado[0] != 'disponible':
                print("✗ Error: La sesión ya no está disponible")
                return False

            # Realizar la reserva
            cursor.execute(
                """UPDATE sesiones
                   SET estudiante_id = ?, estado = 'reservada'
                   WHERE id = ?""",
                (estudiante_id, sesion_id)
            )

            conn.commit()
            conn.close()
            return True
        except Exception as e:
            print(f"✗ Error al reservar sesión: {e}")
            return False

# %% [markdown]
# ## 4. Sistema de Menú Interactivo

# %%
class SistemaTutorias:
    """
    Clase principal que maneja el menú y la lógica de la aplicación.
    """
    def __init__(self):
        self.gestor_db = GestorBaseDatos()

    def leer_entero(self, mensaje: str, minimo: int = None, maximo: int = None) -> Optional[int]:
        """
        Lee un número entero con manejo de errores robusto.
        Punto extra: manejo de errores con try-except
        """
        while True:
            try:
                valor = int(input(mensaje))

                if minimo is not None and valor < minimo:
                    print(f"✗ El valor debe ser mayor o igual a {minimo}")
                    continue

                if maximo is not None and valor > maximo:
                    print(f"✗ El valor debe ser menor o igual a {maximo}")
                    continue

                return valor
            except ValueError:
                print("✗ Error: Debe ingresar un número entero válido")
            except KeyboardInterrupt:
                print("\n✗ Operación cancelada")
                return None

    def mostrar_menu_principal(self):
        """Muestra el menú principal del sistema"""
        print("\n" + "="*60)
        print("   SISTEMA DE GESTIÓN DE TUTORÍAS")
        print("="*60)
        print("1. Registrar nuevo usuario (Tutor o Estudiante)")
        print("2. Tutor: Ofrecer nueva sesión de tutoría")
        print("3. Estudiante: Reservar una sesión disponible")
        print("4. Ver todas las sesiones disponibles")
        print("5. Salir")
        print("="*60)

    def registrar_nuevo_usuario(self):
        """Punto 2: Funcionalidad de registro de usuarios"""
        print("\n--- REGISTRO DE NUEVO USUARIO ---")

        nombre = input("Ingrese el nombre completo: ").strip()
        if not nombre:
            print("✗ El nombre no puede estar vacío")
            return

        correo = input("Ingrese el correo electrónico: ").strip()
        if not correo or "@" not in correo:
            print("✗ Ingrese un correo válido")
            return

        print("\n¿Desea registrarse como tutor o estudiante?")
        print("1. Tutor")
        print("2. Estudiante")

        opcion_rol = self.leer_entero("Seleccione una opción (1-2): ", 1, 2)
        if opcion_rol is None:
            return

        rol = "tutor" if opcion_rol == 1 else "estudiante"

        usuario_id = self.gestor_db.registrar_usuario(nombre, correo, rol)

        if usuario_id:
            print(f"\n✓ Usuario registrado exitosamente con ID: {usuario_id}")
            print(f"  Nombre: {nombre}")
            print(f"  Correo: {correo}")
            print(f"  Rol: {rol.capitalize()}")
        else:
            print("\n✗ No se pudo registrar el usuario")

    def ofrecer_sesion_tutoria(self):
        """Punto 3: Funcionalidad para que un tutor ofrezca una sesión"""
        print("\n--- OFRECER NUEVA SESIÓN DE TUTORÍA ---")

        tutor_id = self.leer_entero("Ingrese su ID de tutor: ", 1)
        if tutor_id is None:
            return

        # Verificar que el usuario existe y es tutor
        usuario = self.gestor_db.obtener_usuario(tutor_id)

        if not usuario:
            print("✗ No se encontró un usuario con ese ID")
            return

        if not isinstance(usuario, Tutor):
            print("✗ El usuario no es un tutor. Solo los tutores pueden ofrecer sesiones.")
            return

        print(f"\n✓ Tutor identificado: {usuario.nombre}")

        # Listar materias disponibles
        materias = self.gestor_db.listar_materias()

        if not materias:
            print("✗ No hay materias disponibles en el sistema")
            return

        print("\n--- MATERIAS DISPONIBLES ---")
        for materia_id, nombre_materia in materias:
            print(f"{materia_id}. {nombre_materia}")

        materia_id = self.leer_entero("\nSeleccione el ID de la materia: ", 1)
        if materia_id is None:
            return

        # Verificar que la materia existe
        materia_valida = any(m[0] == materia_id for m in materias)
        if not materia_valida:
            print("✗ ID de materia no válido")
            return

        print("\nIngrese la fecha y hora de la sesión")
        fecha_hora = input("Formato (YYYY-MM-DD HH:MM): ").strip()

        # Validar formato básico de fecha
        try:
            datetime.strptime(fecha_hora, "%Y-%m-%d %H:%M")
        except ValueError:
            print("✗ Formato de fecha/hora inválido. Use YYYY-MM-DD HH:MM")
            return

        # Crear la sesión
        if self.gestor_db.crear_sesion(tutor_id, materia_id, fecha_hora):
            print("\n✓ Sesión de tutoría creada exitosamente")
            print(f"  Fecha y hora: {fecha_hora}")
            print(f"  Estado: Disponible")
        else:
            print("\n✗ No se pudo crear la sesión")

    def reservar_sesion(self):
        """Punto 4: Funcionalidad para que un estudiante reserve una sesión (Polimorfismo)"""
        print("\n--- RESERVAR SESIÓN DE TUTORÍA ---")

        estudiante_id = self.leer_entero("Ingrese su ID de estudiante: ", 1)
        if estudiante_id is None:
            return

        # Verificar que el usuario existe
        usuario = self.gestor_db.obtener_usuario(estudiante_id)

        if not usuario:
            print("✗ No se encontró un usuario con ese ID")
            return

        # POLIMORFISMO: Usar el método puede_reservar()
        if not usuario.puede_reservar():
            print(f"✗ Los {usuario.rol}es no pueden reservar sesiones.")
            print("  Solo los estudiantes pueden reservar tutorías.")
            return

        print(f"\n✓ Estudiante identificado: {usuario.nombre}")

        # Listar sesiones disponibles
        sesiones = self.gestor_db.listar_sesiones_disponibles()

        if not sesiones:
            print("\n✗ No hay sesiones disponibles en este momento")
            return

        print("\n--- SESIONES DISPONIBLES ---")
        print(f"{'ID':<5} {'Tutor':<20} {'Materia':<20} {'Fecha y Hora':<20}")
        print("-" * 70)
        for sesion_id, tutor, materia, fecha_hora in sesiones:
            print(f"{sesion_id:<5} {tutor:<20} {materia:<20} {fecha_hora:<20}")

        sesion_id = self.leer_entero("\nIngrese el ID de la sesión que desea reservar: ", 1)
        if sesion_id is None:
            return

        # Realizar la reserva
        if self.gestor_db.reservar_sesion(sesion_id, estudiante_id):
            print("\n✓ Sesión reservada exitosamente")
            print("  ¡Nos vemos en la tutoría!")
        else:
            print("\n✗ No se pudo reservar la sesión")

    def ver_sesiones_disponibles(self):
        """Punto 5: Muestra todas las sesiones disponibles"""
        print("\n--- TODAS LAS SESIONES DISPONIBLES ---")

        sesiones = self.gestor_db.listar_sesiones_disponibles()

        if not sesiones:
            print("\n✗ No hay sesiones disponibles en este momento")
            return

        print(f"\n{'ID':<5} {'Tutor':<20} {'Materia':<20} {'Fecha y Hora':<20}")
        print("-" * 70)
        for sesion_id, tutor, materia, fecha_hora in sesiones:
            print(f"{sesion_id:<5} {tutor:<20} {materia:<20} {fecha_hora:<20}")

        print(f"\nTotal de sesiones disponibles: {len(sesiones)}")

    def ejecutar(self):
        """Punto 5: Bucle principal del sistema con menú interactivo"""
        print("\n¡Bienvenido al Sistema de Gestión de Tutorías!")

        while True:
            try:
                self.mostrar_menu_principal()
                opcion = self.leer_entero("Seleccione una opción (1-5): ", 1, 5)

                if opcion is None:
                    continue

                if opcion == 1:
                    self.registrar_nuevo_usuario()
                elif opcion == 2:
                    self.ofrecer_sesion_tutoria()
                elif opcion == 3:
                    self.reservar_sesion()
                elif opcion == 4:
                    self.ver_sesiones_disponibles()
                elif opcion == 5:
                    print("\n¡Gracias por usar el Sistema de Gestión de Tutorías!")
                    print("¡Hasta pronto! 👋\n")
                    break

                input("\nPresione Enter para continuar...")

            except KeyboardInterrupt:
                print("\n\n¿Desea salir del sistema? (s/n): ", end="")
                if input().lower() == 's':
                    print("\n¡Hasta pronto! 👋\n")
                    break
            except Exception as e:
                print(f"\n✗ Error inesperado: {e}")
                input("Presione Enter para continuar...")

# %% [markdown]
# ## 5. Ejecución del Sistema

# %%
# Inicializar y ejecutar el sistema
if __name__ == "__main__":
    sistema = SistemaTutorias()
    sistema.ejecutar()

# %% [markdown]


✓ Base de datos inicializada correctamente
✓ Materias iniciales cargadas

¡Bienvenido al Sistema de Gestión de Tutorías!

   SISTEMA DE GESTIÓN DE TUTORÍAS
1. Registrar nuevo usuario (Tutor o Estudiante)
2. Tutor: Ofrecer nueva sesión de tutoría
3. Estudiante: Reservar una sesión disponible
4. Ver todas las sesiones disponibles
5. Salir
Seleccione una opción (1-5): 2

--- OFRECER NUEVA SESIÓN DE TUTORÍA ---
Ingrese su ID de tutor: 5245
✗ No se encontró un usuario con ese ID

Presione Enter para continuar...

   SISTEMA DE GESTIÓN DE TUTORÍAS
1. Registrar nuevo usuario (Tutor o Estudiante)
2. Tutor: Ofrecer nueva sesión de tutoría
3. Estudiante: Reservar una sesión disponible
4. Ver todas las sesiones disponibles
5. Salir
Seleccione una opción (1-5): 3

--- RESERVAR SESIÓN DE TUTORÍA ---
Ingrese su ID de estudiante: 0000370090
✗ No se encontró un usuario con ese ID

Presione Enter para continuar...

   SISTEMA DE GESTIÓN DE TUTORÍAS
1. Registrar nuevo usuario (Tutor o Estudiante)
2. Tut